# Lab 13 — Capstone — Telco Churn End-to-End + Model Card
**Capstone Track** · Capstone · ~90 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Build full pipeline: clean → encode → train → select
2. Compare two models with cross-validation
3. Write a model card with fairness slice
4. Produce churn_model_card.md deliverable

## Datasets (this folder)
- `Telco-Customer-Churn.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-13-capstone-churn-model-card/lab-13-capstone-churn-model-card.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Paste the lab token from the ViLabs page into `VL_TOKEN` in **Cell 0**, then run that cell first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`Telco-Customer-Churn.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-13-capstone-churn-model-card"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-13-capstone-churn-model-card/bundle/dataset.zip"
NEED = ["Telco-Customer-Churn.csv"]  # unzipped files used by the code below


import json as _json
import urllib.parse as _urlparse
import urllib.request as _urlrequest

VLABS_API = "https://prod-api.vilabs.app"
VL_TOKEN = ""  # paste the lab token shown on the ViLabs page after Start Lab

def vlabs_submit():
    """Validation function. Do not modify."""
    import inspect
    token = str(VL_TOKEN or "").strip()
    if not token:
        print("ViLabs: paste the lab token into VL_TOKEN in Cell 0, then re-run this cell.")
        return None
    names = ["pr_auc_gap", "f1_threshold"]
    answers = {}
    missing = []
    for name in names:
        fn = globals().get(name)
        if not callable(fn):
            missing.append(name)
            continue
        answers[name] = inspect.getsource(fn)
    if missing:
        print("ViLabs: write these functions before validating:", ", ".join(missing))
        return None
    base = VLABS_API.rstrip("/") + "/colab/labs/" + LAB_ID + "/session"
    ua = {"User-Agent": "Mozilla/5.0 (compatible; ViLabs-Colab/1.0)"}

    def _read(req):
        try:
            with _urlrequest.urlopen(req, timeout=90) as resp:
                return _json.loads(resp.read().decode())
        except _urlrequest.HTTPError as exc:
            print("ViLabs HTTP", exc.code, exc.read().decode(errors="replace")[:300])
            return None

    payload = _json.dumps({"token": token, "answers": answers}).encode()
    result = _read(_urlrequest.Request(
        base + "/validate/",
        data=payload,
        headers={"Content-Type": "application/json", **ua},
    ))
    if result is None:
        return None
    print("ViLabs validated", result.get("validated"))
    for row in result.get("variables") or []:
        print(row.get("name"), "pass" if row.get("ok") else "fail")
    return result

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Capstone: Telco Churn End-to-End + Model Card

> **Scenario:** The retention team at a telecom wants a churn model they can actually trust: clean the raw contract data, encode it properly, train and compare two models, pick a decision threshold, explain the drivers, and document everything in a model card. You will run the full pipeline end-to-end and leave with a deployable artifact and its documentation.
>
> **You will learn:** messy-data cleaning (blank `TotalCharges`), binary + one-hot encoding, pipeline-based preprocessing, model selection by PR-AUC, threshold tuning, permutation importance, fairness slicing, model-card documentation.
> **Time:** ~90 minutes. **Level:** Capstone. **Needs:** pandas + sklearn. **Env:** 🟢 Colab only.

### Mental Map

| Section | Key idea | sklearn / pandas tool |
|---|---|---|
| 1 | Clean the raw file | `pd.to_numeric(errors="coerce")` |
| 2 | Encode without leakage | `OneHotEncoder`, `df.map` |
| 3 | Train two models in pipelines | `Pipeline`, `ColumnTransformer` |
| 4 | Select by PR-AUC; tune the threshold | `precision_recall_curve` |
| 5 | Explain with permutation importance | `permutation_importance` |
| 6 | Fairness slices + model card | `groupby`, file write |

---

## 1. Clean the Raw File


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("Telco-Customer-Churn.csv")
print(df.shape)
print(df.dtypes)

df = df.drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("blank TotalCharges coerced to NaN:", int(df["TotalCharges"].isna().sum()))
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)

df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
print(f"churn rate: {df['Churn'].mean():.4f}")
print(df["Churn"].value_counts())


**What to notice:**
1. `TotalCharges` is stored as text because 11 new customers have a blank — `pd.to_numeric(errors="coerce")` turns blanks into NaN so they can be dropped.
2. `customerID` is a row identifier, not a feature — dropping it prevents the model from memorising customers.
3. The churn rate is 0.2658 — imbalanced, so PR-AUC is again the headline metric.

> **Pitfall:** `SeniorCitizen` is already 0/1 — do not one-hot encode it, or you'll create a redundant column.

---

## 2. Encode Without Leakage


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
for c in binary_cols:
    df[c] = df[c].map({"No": 0, "Yes": 1})

cat_cols = ["gender", "MultipleLines", "InternetService", "OnlineSecurity",
            "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
            "StreamingMovies", "Contract", "PaymentMethod"]
num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"] + binary_cols

X = df.drop(columns=["Churn"])
y = df["Churn"]

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
])
print("numeric:", len(num_cols), "| categorical:", len(cat_cols), "| binary-mapped:", len(binary_cols))


**What to notice:**
1. `drop="first"` avoids the dummy-variable trap (perfect collinearity) — harmless for trees, helpful for linear models.
2. `handle_unknown="ignore"` makes the encoder safe on new categories at serving time.
3. Binary Yes/No columns are mapped to 0/1 and standardised with the numerics — no need to one-hot them.

> **Pitfall:** fit the `ColumnTransformer` **inside** a pipeline on training data only. Fitting it on the full dataset leaks category statistics into training. Any column the `ColumnTransformer` doesn't list is silently dropped — check `num_cols + cat_cols` covers everything.

---

## 3. Train Two Models in Pipelines


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

log_pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=42))])
gbm_pipe = Pipeline([("pre", pre), ("clf", GradientBoostingClassifier(random_state=42))])

log_pipe.fit(X_train, y_train)
gbm_pipe.fit(X_train, y_train)

for name, pipe in [("Logistic", log_pipe), ("GBM", gbm_pipe)]:
    p = pipe.predict_proba(X_test)[:, 1]
    print(f"{name:9s} ROC-AUC: {roc_auc_score(y_test, p):.4f}  "
          f"PR-AUC: {average_precision_score(y_test, p):.4f}  "
          f"F1@0.5: {f1_score(y_test, (p >= 0.5).astype(int)):.4f}")


**What to notice:**
1. The pipeline guarantees the encoder/scaler is refit on training folds only — no leakage, even under cross-validation.
2. GBM wins on PR-AUC (0.662 vs 0.624); logistic wins on F1@0.5 (0.611 vs 0.587). The selection metric must be chosen before looking.
3. F1 at the default 0.5 threshold is a quick sanity check, not the final operating point.

> **Pitfall:** `predict_proba` from a pipeline returns probabilities for **all** classes — always slice `[:, 1]` for the positive class.

---

## 4. Select the Model and Tune the Threshold


In [ ]:
from sklearn.metrics import precision_recall_curve

p_gbm = gbm_pipe.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, p_gbm)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
best_i = np.argmax(f1)
print(f"best threshold: {thr[best_i]:.4f}  precision: {prec[best_i]:.4f}  "
      f"recall: {rec[best_i]:.4f}  F1: {f1[best_i]:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thr, prec[:-1], label="precision")
ax.plot(thr, rec[:-1], label="recall")
ax.axvline(thr[best_i], color="k", ls="--", lw=1, label=f"best F1 @ {thr[best_i]:.2f}")
ax.set_xlabel("threshold"); ax.set_ylabel("score"); ax.legend()
fig.tight_layout()
fig.savefig("lab13_threshold.png", dpi=120)
from IPython.display import display
display(fig)
print("saved lab13_threshold.png")


**What to notice:**
1. The F1-optimal threshold is 0.395 — **below** the default 0.5, as usual for imbalanced data.
2. Precision and recall trade off along the curve; the threshold you pick encodes how much a false alarm costs vs a missed churner.
3. Choosing the threshold on the **test** set is itself a mild leak — in production, tune it on a validation split.

> **Pitfall:** never tune the threshold on the same data you report final metrics on. Split train/validation/test and keep test sacred.

---

## 5. Explain with Permutation Importance


In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(gbm_pipe, X_test, y_test, n_repeats=10,
                            random_state=42, scoring="average_precision")
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
print(imp.head(10).round(4))


**What to notice:**
1. `tenure` dominates (importance ≈ 0.19), then `MonthlyCharges` and `Contract` — short-tenure, month-to-month customers on high plans churn most.
2. Permutation importance is computed on test data, so it reflects real generalisation, not training-set artefacts.
3. The index is the **raw** input columns (19 of them) — permutation importance permutes the model inputs, not the one-hot-expanded features.

> **Pitfall:** impurity-based importances are computed on training data and favour continuous features; permutation importance on held-out data is the trustworthy default.

---

## 6. Fairness Slices and the Model Card


In [ ]:
def slice_table(col):
    out = df.groupby(col)["Churn"].agg(["count", "mean"])
    out.columns = ["n", "churn_rate"]
    return out.round(4)

for col in ["gender", "SeniorCitizen", "Contract", "InternetService"]:
    print(f"--- {col} ---")
    print(slice_table(col))


**What to notice:**
1. Churn varies sharply by slice: month-to-month customers churn at 0.427 vs 0.028 for two-year contracts; senior citizens at 0.417 vs 0.237.
2. Gender is nearly balanced (0.270 vs 0.262) — a slice where the model should behave similarly.
3. These tables are descriptive; they flag where to monitor, they don't prove unfairness.

> **Pitfall:** a fairness table on the whole dataset mixes "who churns" with "who the model flags". Always pair it with slice-wise precision/recall before claiming bias.

Now write the model card. The code computes every number from the fitted pipeline and writes `churn_model_card.md`:


In [ ]:
p_final = gbm_pipe.predict_proba(X_test)[:, 1]
y_pred = (p_final >= thr[best_i]).astype(int)

rows_md = []
for col in ["gender", "SeniorCitizen", "Contract"]:
    t = df.groupby(col)["Churn"].agg(["count", "mean"])
    for idx, row in t.iterrows():
        rows_md.append(f"| {col} = {idx} | {int(row['count'])} | {row['mean']:.4f} |")
table_md = "\n".join(rows_md)

card = f"""# Model Card: Telco Churn Predictor

## Intended Use
Predict the probability that a customer will churn within the billing cycle,
so the retention team can prioritise outreach. Not for credit, pricing, or
employment decisions.

## Training Data
- Source: Telco-Customer-Churn.csv ({len(df)} rows after cleaning)
- Split: 75% train / 25% test, stratified on Churn (random_state=42)
- Features: {len(num_cols)} numeric (standardised), {len(cat_cols)} categorical
  (one-hot, drop-first), {len(binary_cols)} binary-mapped
- Positive rate: {y.mean():.3f}

## Performance Metrics (test set, n={len(y_test)})
- ROC-AUC: {roc_auc_score(y_test, p_final):.4f}
- PR-AUC: {average_precision_score(y_test, p_final):.4f}
- Decision threshold: {thr[best_i]:.4f} (F1-optimal)
- Precision / Recall / F1 at threshold: {prec[best_i]:.4f} / {rec[best_i]:.4f} / {f1[best_i]:.4f}

## Fairness
Churn rates differ across slices (see table below). Monitor these slices
monthly; investigate any slice where precision drops below the global average.

| Slice | n | churn rate |
|---|---|---|
{table_md}

## Limitations
- TotalCharges blanks (11 rows) were dropped; new customers are under-represented.
- Threshold tuned on the test set — re-tune on a fresh validation split before deployment.
- Model does not capture seasonality or competitor actions.
- Fairness slices are descriptive; no causal claim is made.
"""

with open("churn_model_card.md", "w", encoding="utf-8") as f:
    f.write(card)
print("wrote churn_model_card.md")
print(card[:600])


**What to notice:**
1. Every number in the card is computed from the fitted pipeline — no hand-typed metrics that can drift out of sync.
2. The five required sections are all present: Intended Use, Training Data, Performance Metrics, Fairness, Limitations.
3. The fairness table is generated from the data, so it stays correct if the dataset is refreshed.

> **Pitfall:** a model card with hardcoded metrics rots the moment the model is retrained. Generate it from the pipeline, as above.

---

## Exercises

Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell.

### Exercise 1 — Reproduce the metrics with the new encoding
Using the pipeline encoding from section 2, report test ROC-AUC and PR-AUC for both models.


**Follow-up:** Which model has higher PR-AUC?

### Exercise 2 — Fairness slice table
Build a table of churn rate by `Contract` and by `gender`.


**Follow-up:** Which slice has the highest churn rate?

### Exercise 3 — Fill the model card
Run the model-card writer and verify the file exists with all five required sections.


**Follow-up:** What is the F1-optimal threshold?

---


In [ ]:
def pr_auc_gap():
    """pr auc gap."""
    pass


In [ ]:
def f1_threshold():
    """f1 threshold."""
    pass


In [ ]:
vlabs_submit()


### What to learn next
- Add slice-wise precision/recall to the fairness table for a sharper bias audit.
- Replace GBM with `HistGradientBoostingClassifier` and compare training time.
- Package the pipeline with `joblib.dump` and write a small inference script.

*Files in this folder: Telco-Customer-Churn.csv.*


---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
